# **Project Name**    - DeepFER: Facial Emotion Recognition Using Deep Learning



##### **Project Type**    - Classification (deep learning, computer vision: 7 facial emotions)
##### **Contribution**    - Individual
##### **Team Member 1 -** Taha Saifee

# **Project Summary -**

Answer Here.

# **GitHub Link -**

https://github.com/Tahascommit/deepfer-emotion-recognition

# **Problem Statement**


Answer Here.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required. 
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits. 
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule. 

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

Answer Here.

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import os
import sys
import json
import time
import copy
import zipfile
import warnings
import importlib
import itertools
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import cv2
from skimage.feature import hog
import torch
import torch.nn.functional as F
import torchvision

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

FIG = Path("outputs/figures")
FIG.mkdir(parents=True, exist_ok=True)
Path("models").mkdir(exist_ok=True)


def save(name):
    """Save the current chart to outputs/figures (used in the README and the app) and show it."""
    plt.savefig(FIG / name, dpi=110, bbox_inches="tight")
    plt.show()


# QUICK is only for a fast check that every cell runs (tiny subsets and 1-2 epochs). The real run has it off.
QUICK = os.environ.get("DEEPFER_QUICK") == "1"
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| opencv", cv2.__version__, "| quick test run" if QUICK else "| full run")

### Dataset Loading

#### The shared code (`deepfer.py`)
The loading, cleaning, augmentation, model, training, explanation and face detection code is written to `deepfer.py`. The Streamlit app and the live webcam script import the same file, so they prepare a face in exactly the same way as this notebook.

In [ ]:
%%writefile deepfer.py
# DeepFER: shared code for the notebook, the Streamlit app and the live webcam script.
# The notebook writes this file, so the app and the webcam script use exactly the same steps.
import copy
import hashlib
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
IMG = 48                                           # every image in the dataset is 48x48 grayscale
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_device():
    """NVIDIA GPU on Colab, the Apple GPU (MPS) on a Mac, otherwise the CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def sync(device):
    """Wait for the GPU to finish, so timings are real."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def log_line(msg):
    """Print a training line. If the DEEPFER_LOG variable names a file, the line is also added to it,
    which makes it easy to follow a long training run from outside the notebook."""
    import os
    print(msg)
    if os.environ.get("DEEPFER_LOG"):
        with open(os.environ["DEEPFER_LOG"], "a") as fh:
            fh.write(msg + "\n")


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)


# ---------------------------------------------------------------- data
def load_images(root):
    """Read every jpg under root/<folder>/<emotion>/ into a table plus one uint8 array (n, 48, 48)."""
    rows, pix = [], []
    for folder in ("train", "validation"):
        for label in CLASSES:
            for f in sorted(Path(root, folder, label).glob("*.jpg")):
                img = Image.open(f)
                rows.append((f"{folder}/{label}/{f.name}", folder, label, img.size[0], img.size[1], img.mode))
                pix.append(np.array(img.convert("L").resize((IMG, IMG))))
    df = pd.DataFrame(rows, columns=["file", "folder", "label", "width", "height", "mode"])
    return df, np.stack(pix)


def hog_lbp_features(X, use_clahe=False):
    """Handcrafted features for the classic baseline: HOG (edge directions in 8x8 cells, 900 numbers)
    plus LBP texture histograms in a 4x4 grid of regions (160 numbers)."""
    from skimage.feature import hog, local_binary_pattern
    out = np.empty((len(X), 1060), dtype=np.float32)
    for k, a in enumerate(X):
        if use_clahe:
            a = equalize(a)
        h = hog(a, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True)
        lbp = local_binary_pattern(a, P=8, R=1, method="uniform").astype(int)
        hist = [np.bincount(lbp[r:r + 12, c:c + 12].ravel(), minlength=10) / 144 for r in range(0, 48, 12) for c in range(0, 48, 12)]
        out[k] = np.concatenate([h, np.concatenate(hist)])
    return out


def image_stats(X, chunk=4000):
    """Simple numbers per image: brightness, contrast, sharpness, edge strength by face region and more."""
    out = []
    for s in range(0, len(X), chunk):
        x = X[s:s + chunk].astype(np.float32)
        gy, gx = np.gradient(x, axis=(1, 2))
        edge = np.hypot(gx, gy)
        lap = x[:, :-2, 1:-1] + x[:, 2:, 1:-1] + x[:, 1:-1, :-2] + x[:, 1:-1, 2:] - 4 * x[:, 1:-1, 1:-1]
        flat = []
        for a in X[s:s + chunk]:
            h = np.bincount(a.ravel(), minlength=256)
            m = h.argmax()
            flat.append(h[max(0, m - 3):m + 4].sum() / a.size)
        p = np.stack([np.bincount(a.ravel(), minlength=256) for a in X[s:s + chunk]]) / (IMG * IMG)
        ent = -(p * np.log2(np.where(p > 0, p, 1))).sum(1)
        out.append(pd.DataFrame({
            "brightness": x.mean((1, 2)),
            "contrast": x.std((1, 2)),
            "sharpness": lap.var((1, 2)),
            "entropy": ent,
            "edge_eyes": edge[:, 10:24, 6:42].mean((1, 2)),     # band with the brows and eyes
            "edge_nose": edge[:, 22:32, 14:34].mean((1, 2)),
            "edge_mouth": edge[:, 31:45, 10:38].mean((1, 2)),   # band with the mouth
            "asymmetry": np.abs(x - x[:, :, ::-1]).mean((1, 2)),
            "flat_share": flat,
            "hash": [hashlib.md5(a.tobytes()).hexdigest() for a in X[s:s + chunk]],
        }))
    return pd.concat(out, ignore_index=True)


def clean_dataset(df, non_face_files, blank_contrast=5.0):
    """Mark the rows to drop, in this order: blank images, non-face images, copies with conflicting
    labels, training copies of a test image, and repeated copies inside the same folder."""
    df = df.copy()
    df["drop_reason"] = ""
    df.loc[df["contrast"] < blank_contrast, "drop_reason"] = "blank image"
    df.loc[(df["drop_reason"] == "") & df["file"].isin(set(non_face_files)), "drop_reason"] = "not a face"
    keep = df["drop_reason"] == ""
    n_labels = df[keep].groupby("hash")["label"].transform("nunique")
    df.loc[n_labels[n_labels > 1].index, "drop_reason"] = "same image, different labels"
    keep = df["drop_reason"] == ""
    test_hashes = set(df.loc[keep & (df["folder"] == "validation"), "hash"])
    leak = keep & (df["folder"] == "train") & df["hash"].isin(test_hashes)
    df.loc[leak, "drop_reason"] = "copy of a test image"
    keep = df["drop_reason"] == ""
    dup = keep & df[keep].duplicated(["folder", "hash"]).reindex(df.index, fill_value=False)
    df.loc[dup, "drop_reason"] = "duplicate in the same folder"
    return df


# ---------------------------------------------------------------- augmentation on the GPU
def augment(x, rot=15, scale=0.12, shift=0.1, flip=True, bright=0.2, contrast=0.3, erase_p=0.25):
    """Random flip, rotation, zoom, shift, lighting change and a cut-out patch, for a whole batch at once.
    x: (B, 1, 48, 48) floats in [0, 1] on any device. Every image gets its own random values."""
    B, dev = x.size(0), x.device
    u = lambda *s: torch.rand(*s, device=dev) * 2 - 1           # uniform in [-1, 1]
    ang = u(B) * math.radians(rot)
    sc = 1 + u(B) * scale
    fl = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0) if flip else torch.ones(B, device=dev)
    t = u(B, 2) * shift
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos * fl, -sin, t[:, 0]], 1),
                         torch.stack([sin * fl, cos, t[:, 1]], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False),
                      padding_mode="border", align_corners=False)
    m = x.mean((2, 3), keepdim=True)
    x = ((x - m) * (1 + u(B, 1, 1, 1) * contrast) + m + u(B, 1, 1, 1) * bright).clamp(0, 1)
    if erase_p > 0:                                              # random erasing: hide a patch
        H = x.size(-1)
        h = (torch.rand(B, device=dev) * 0.25 + 0.1) * H
        w = (torch.rand(B, device=dev) * 0.25 + 0.1) * H
        y0 = torch.rand(B, device=dev) * (H - h)
        x0 = torch.rand(B, device=dev) * (H - w)
        r = torch.arange(H, device=dev).float()
        my = (r.view(1, H) >= y0.view(B, 1)) & (r.view(1, H) < (y0 + h).view(B, 1))
        mx = (r.view(1, H) >= x0.view(B, 1)) & (r.view(1, H) < (x0 + w).view(B, 1))
        mask = (my.view(B, H, 1) & mx.view(B, 1, H)) & (torch.rand(B, 1, 1, device=dev) < erase_p)
        x = torch.where(mask.unsqueeze(1), torch.full_like(x, 0.5), x)
    return x


# ---------------------------------------------------------------- models
class InputAdapter(nn.Module):
    """Every saved model takes the same input: grayscale 48x48 in [0, 1]. This layer resizes it,
    copies it to 3 channels for the ImageNet models and normalises it."""

    def __init__(self, size, rgb, mean, std):
        super().__init__()
        self.size, self.rgb = size, rgb
        c = 3 if rgb else 1
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, c, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, c, 1, 1))

    def forward(self, x):
        if self.size != x.size(-1):
            x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", align_corners=False)
        if self.rgb:
            x = x.expand(-1, 3, -1, -1)
        return (x - self.mean) / self.std


class SqueezeExcite(nn.Module):
    """Channel attention: the block learns which feature maps matter for the current face."""

    def __init__(self, c, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c // r, 1), nn.ReLU(inplace=True),
                                nn.Conv2d(c // r, c, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(x)


class ResBlock(nn.Module):
    """Two 3x3 convolutions with batch norm, squeeze-excitation and a shortcut connection."""

    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), SqueezeExcite(cout))
        self.skip = nn.Identity() if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        return F.relu(self.body(x) + self.skip(x))


class DeepFERNet(nn.Module):
    """Custom CNN for 48x48 faces: a stem at full resolution, then three residual stages that halve
    the size (48 -> 24 -> 12 -> 6) while the channels grow (64 -> 128 -> 256 -> 512)."""

    def __init__(self, n_classes=7, width=64, dropout=0.4):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(1, w // 2, 3, 1, 1, bias=False), nn.BatchNorm2d(w // 2), nn.ReLU(inplace=True),
                                  nn.Conv2d(w // 2, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True))
        self.stage1 = nn.Sequential(ResBlock(w, 2 * w, 2), ResBlock(2 * w, 2 * w))
        self.stage2 = nn.Sequential(ResBlock(2 * w, 4 * w, 2), ResBlock(4 * w, 4 * w))
        self.stage3 = ResBlock(4 * w, 8 * w, 2)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(8 * w, n_classes))

    def forward(self, x):
        return self.head(self.stage3(self.stage2(self.stage1(self.stem(x)))))


def build_model(name, pretrained=True, dropout=0.3, size=128, pixel_mean=0.5, pixel_std=0.25):
    """deepfer_cnn (from scratch), resnet18 or efficientnet_b0 (ImageNet weights, new 7-class head)."""
    import torchvision
    if name == "deepfer_cnn":
        net = DeepFERNet(len(CLASSES), dropout=max(dropout, 0.4))
        return nn.Sequential(InputAdapter(IMG, False, [pixel_mean], [pixel_std]), net)
    if name == "resnet18":
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        net = torchvision.models.resnet18(weights=w)
        net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(net.fc.in_features, len(CLASSES)))
    elif name == "efficientnet_b0":
        w = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        net = torchvision.models.efficientnet_b0(weights=w)
        net.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(net.classifier[1].in_features, len(CLASSES)))
    else:
        raise ValueError(name)
    return nn.Sequential(InputAdapter(size, True, IMAGENET_MEAN, IMAGENET_STD), net)


def head_and_body(model):
    """Split the parameters into the new classifier head and the rest (the pre-trained body)."""
    net = model[1]
    head = net.head if isinstance(net, DeepFERNet) else (net.fc if hasattr(net, "fc") else net.classifier)
    head_ids = {id(p) for p in head.parameters()}
    return list(head.parameters()), [p for p in net.parameters() if id(p) not in head_ids]


def cam_layer(model):
    """The last convolution block, used by Grad-CAM."""
    net = model[1]
    if isinstance(net, DeepFERNet):
        return net.stage3
    return net.layer4 if hasattr(net, "layer4") else net.features[-1]


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------- training
def to_batch(xu8, device):
    """uint8 (B, 48, 48) -> float (B, 1, 48, 48) in [0, 1]."""
    return xu8.to(device).float().div(255).unsqueeze(1)


def amp_ok(device):
    return device.type in ("cuda", "mps")


def amp_dtype(device):
    """Half precision for speed: bfloat16 on the Apple GPU (same number range as float32, so no loss scaling
    is needed), float16 with loss scaling on an NVIDIA GPU."""
    return torch.bfloat16 if device.type == "mps" else torch.float16


@torch.no_grad()
def predict_logits(model, X, device, batch=512, tta=False):
    """Raw scores for a uint8 tensor of faces. tta=True also scores the mirrored face and averages."""
    model.eval()
    out = []
    for s in range(0, len(X), batch):
        x = to_batch(X[s:s + batch], device)
        with torch.autocast(device.type, dtype=amp_dtype(device), enabled=amp_ok(device)):
            z = model(x).float()
            if tta:
                z = (z + model(torch.flip(x, dims=[3])).float()) / 2
        out.append(z.cpu())
    return torch.cat(out)


def scores(y, pred):
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    return {"accuracy": accuracy_score(y, pred),
            "macro precision": precision_score(y, pred, average="macro", zero_division=0),
            "macro recall": recall_score(y, pred, average="macro", zero_division=0),
            "macro F1": f1_score(y, pred, average="macro", zero_division=0),
            "weighted F1": f1_score(y, pred, average="weighted", zero_division=0)}


def train_model(model, Xtr, ytr, Xva, yva, device, epochs=30, batch=128, lr=1e-3, lr_body=None,
                weight_decay=1e-4, label_smoothing=0.1, class_weights=None, patience=6,
                freeze_body=False, aug=None, seed=42, log=log_line):
    """Mini-batch training with AdamW, a one-cycle learning rate (warm-up then cosine decay),
    mixed precision on the GPU (bfloat16 on Apple, float16 on NVIDIA), GPU augmentation and early stopping
    on validation macro F1.
    The weights of the best epoch are put back at the end."""
    set_seed(seed)
    model = model.to(device)
    head, body = head_and_body(model)
    for p in body:
        p.requires_grad_(not freeze_body)
    groups = [{"params": head, "lr": lr}]
    if not freeze_body:
        groups.append({"params": body, "lr": lr_body or lr})
    opt = torch.optim.AdamW(groups, weight_decay=weight_decay)
    steps = math.ceil(len(Xtr) / batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=epochs * steps,
                                                pct_start=0.15, div_factor=10, final_div_factor=100)
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    w = None if class_weights is None else torch.tensor(class_weights, dtype=torch.float32, device=device)
    ytr = ytr.to(Xtr.device)
    aug = aug or {}
    best, best_state, wait, hist = -1, None, 0, []
    g = torch.Generator().manual_seed(seed)
    for ep in range(1, epochs + 1):
        model.train()
        if freeze_body:                      # keep the frozen body's batch-norm statistics fixed
            for m in model[1].modules():
                if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
                    m.eval()
        t0, tot, n = time.time(), 0.0, 0
        order = torch.randperm(len(Xtr), generator=g).to(Xtr.device)
        for s in range(0, len(Xtr), batch):
            idx = order[s:s + batch]
            x = augment(to_batch(Xtr[idx], device), **aug)
            yb = ytr[idx].to(device)
            with torch.autocast(device.type, dtype=amp_dtype(device), enabled=amp_ok(device)):
                loss = F.cross_entropy(model(x).float(), yb, weight=w, label_smoothing=label_smoothing)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if sched.last_epoch < sched.total_steps - 1:
                sched.step()
            tot += loss.item() * len(idx)
            n += len(idx)
        zva = predict_logits(model, Xva, device)
        va_loss = F.cross_entropy(zva, yva).item()
        sc = scores(yva.numpy(), zva.argmax(1).numpy())
        hist.append({"epoch": ep, "train loss": tot / n, "val loss": va_loss, "val accuracy": sc["accuracy"],
                     "val macro F1": sc["macro F1"], "lr": opt.param_groups[0]["lr"], "seconds": time.time() - t0})
        log(f"epoch {ep:2d}  train loss {tot / n:.3f}  val loss {va_loss:.3f}  val acc {sc['accuracy']:.3f}  "
            f"val macro F1 {sc['macro F1']:.3f}  ({time.time() - t0:.0f}s)")
        if sc["macro F1"] > best + 1e-4:
            best, wait = sc["macro F1"], 0
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience:
                log(f"early stop: no gain for {patience} epochs, best val macro F1 {best:.3f}")
                break
    model.load_state_dict(best_state)
    for p in model.parameters():
        p.requires_grad_(True)
    return model, pd.DataFrame(hist)


def fit_temperature(logits, y):
    """Temperature scaling: one number that makes the probabilities honest (calibrated) on validation."""
    grid = np.linspace(0.5, 3.0, 51)
    nll = [F.cross_entropy(logits / t, y).item() for t in grid]
    return float(grid[int(np.argmin(nll))])


def save_model(model, path, name, extra=None):
    torch.save({"name": name, "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "extra": extra or {}}, path)


def load_model(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    ex = ck.get("extra", {})
    model = build_model(ck["name"], pretrained=False, size=ex.get("size", 128),
                        pixel_mean=ex.get("pixel_mean", 0.5), pixel_std=ex.get("pixel_std", 0.25))
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval(), ex


# ---------------------------------------------------------------- explainability
def grad_cam(model, x, target=None):
    """Grad-CAM heat map: which parts of the face pushed the model towards the class.
    x: (B, 1, 48, 48) on the model's device. Returns maps (B, 48, 48) in [0, 1] and the classes used."""
    model.eval()
    layer, store = cam_layer(model), {}

    def keep(module, inputs, out):
        store["a"] = out
        out.register_hook(lambda grad: store.__setitem__("g", grad))

    hook = layer.register_forward_hook(keep)
    x = x.clone().requires_grad_(True)
    z = model(x)
    cls = z.argmax(1) if target is None else torch.as_tensor(target, device=x.device).view(-1)
    model.zero_grad()
    z.gather(1, cls.view(-1, 1)).sum().backward()
    hook.remove()
    a, g = store["a"], store["g"]
    cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam / (cam.amax((1, 2), keepdim=True) + 1e-8)
    return cam.detach().cpu().numpy(), cls.cpu().numpy()


# ---------------------------------------------------------------- real time: faces in a photo or a video frame
class FaceFinder:
    """OpenCV's Haar cascade face detector (it ships with OpenCV, so nothing to download)."""

    def __init__(self):
        import cv2
        self.cv2 = cv2
        self.det = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    def find(self, gray, scale_to=480, min_face=40):
        """Boxes (x, y, w, h) of the faces in a grayscale image. Large images are shrunk first for speed."""
        f = min(1.0, scale_to / max(gray.shape))
        small = self.cv2.resize(gray, None, fx=f, fy=f) if f < 1 else gray
        boxes = self.det.detectMultiScale(small, scaleFactor=1.1, minNeighbors=5,
                                          minSize=(int(min_face * f), int(min_face * f)))
        return [tuple(int(v / f) for v in b) for b in boxes] if len(boxes) else []


def crop_face(gray, box, scale=1.03, shift=0.02):
    """Cut the face out the same way the training faces were cut and make it 48x48. Measured on enlarged
    test faces: the dataset crop is almost exactly OpenCV's face box, 3% larger and 2% further right and down."""
    import cv2
    x, y, w, h = box
    size = int(round(w * scale))
    x0, y0 = int(round(x + shift * w)), int(round(y + shift * w))
    pad = size                                              # pad with edge pixels in case the face touches the border
    big = cv2.copyMakeBorder(gray, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
    face = big[y0 + pad:y0 + pad + size, x0 + pad:x0 + pad + size]
    return cv2.resize(face, (IMG, IMG), interpolation=cv2.INTER_AREA)


def equalize(img):
    """CLAHE histogram equalisation: evens out dark or flat lighting, tile by tile."""
    import cv2
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(img)


class EmotionPredictor:
    """The deployed model: one or more saved networks (averaged), mirrored-face averaging and temperature,
    exactly as chosen in the notebook and written to models/model_card.json."""

    def __init__(self, model_dir="models", device="cpu"):
        import json
        self.card = json.loads(Path(model_dir, "model_card.json").read_text())
        self.device = torch.device(device)          # the CPU by default: fast enough for a few faces and works everywhere
        self.models = [load_model(Path(model_dir, f), self.device)[0] for f in self.card["members"]]
        self.temperature = self.card["temperature"]
        self.tta = self.card["tta"]

    @torch.no_grad()
    def logits(self, x):
        z = 0
        for m in self.models:
            z = z + m(x).float()
            if self.tta:
                z = z + m(torch.flip(x, dims=[3])).float()
        return z / (len(self.models) * (2 if self.tta else 1))

    def predict(self, faces):
        """Probabilities (n, 7) for a list or array of 48x48 uint8 faces."""
        if len(faces) == 0:
            return np.zeros((0, len(CLASSES)))
        x = to_batch(torch.from_numpy(np.stack(faces)), self.device)
        return torch.softmax(self.logits(x) / self.temperature, 1).cpu().numpy()

    def explain(self, face):
        """Grad-CAM heat map of the first network for one 48x48 face."""
        cam, _ = grad_cam(self.models[0], to_batch(torch.from_numpy(face[None]), self.device))
        return cam[0]


@torch.no_grad()
def predict_faces(model, faces, device, temperature=1.0):
    """Emotion probabilities for a list of 48x48 uint8 faces."""
    if len(faces) == 0:
        return np.zeros((0, len(CLASSES)))
    x = to_batch(torch.from_numpy(np.stack(faces)), device)
    model.eval()
    return torch.softmax(model(x).float() / temperature, 1).cpu().numpy()


def analyse_image(gray, finder, predictor):
    """The real-time pipeline for one photo or video frame: find faces, crop them like the training faces,
    and score them all in one batch. Returns the boxes, the 48x48 faces and the probabilities."""
    boxes = finder.find(gray)
    faces = [crop_face(gray, b) for b in boxes]
    return boxes, faces, predictor.predict(faces)


In [ ]:
# Load Dataset
# The images sit in data/images/<train|validation>/<emotion>/*.jpg. On Colab that folder is missing,
# so the zip of the images is downloaded from the project's GitHub repo and unpacked first.
DATA = Path("data/images")
REPO_RAW = "https://raw.githubusercontent.com/Tahascommit/deepfer-emotion-recognition/main"
if not DATA.exists():
    Path("data").mkdir(exist_ok=True)
    for name in ["face_emotion_images.zip", "non_face_images.csv"]:
        if not Path("data", name).exists():
            print("downloading", name)
            urllib.request.urlretrieve(f"{REPO_RAW}/data/{name}", f"data/{name}")
    with zipfile.ZipFile("data/face_emotion_images.zip") as z:
        z.extractall("data")

import deepfer
importlib.reload(deepfer)                 # pick up the file written above if the cell is run again
CLASSES = deepfer.CLASSES
DEVICE = deepfer.get_device()

t0 = time.time()
df, X = deepfer.load_images(DATA)         # df: one row per image, X: all pixels as uint8 (n, 48, 48)
print(f"{len(df):,} images loaded in {time.time() - t0:.1f}s into one array of shape {X.shape} ({X.nbytes / 1e6:.0f} MB)")
print("device used for training:", DEVICE, "(mps = Apple GPU, cuda = NVIDIA GPU)")

### Dataset First View

In [ ]:
# Dataset First Look
display(df.head(8))
rs = np.random.RandomState(SEED)
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, i in zip(axes, rs.choice(len(df), 10, replace=False)):
    ax.imshow(X[i], cmap="gray", vmin=0, vmax=255)
    ax.set_title(df.label[i], fontsize=10)
    ax.axis("off")
plt.suptitle("Ten random images with the emotion of their folder", y=1.1)
plt.show()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"images (rows): {len(df):,}   columns in the table: {df.shape[1]}   "
      f"pixels per image: {X.shape[1]} x {X.shape[2]} = {X.shape[1] * X.shape[2]:,}")
display(pd.crosstab(df["label"], df["folder"], margins=True, margins_name="total"))

### Dataset Information

In [ ]:
# Dataset Info
df.info()
print()
print(df[["width", "height", "mode"]].value_counts().rename("images"))
print(f"\npixel values from {X.min()} to {X.max()}, stored as {X.dtype}")

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
# Numbers per image (brightness, contrast, sharpness, edge strength by face region ...) and an md5 hash
# of the pixels, so two files with exactly the same picture get the same hash.
stats_df = deepfer.image_stats(X)
df = pd.concat([df, stats_df], axis=1)

print("repeated file paths:", df["file"].duplicated().sum())
by_hash = df.groupby("hash")
size = by_hash.size()
print(f"pixel-identical pictures: {(size > 1).sum():,} groups with {int((size[size > 1] - 1).sum()):,} extra copies")
print("groups that appear in both the train and the validation folder:", int((by_hash["folder"].nunique() > 1).sum()))
print("groups whose copies carry different emotion labels:", int((by_hash["label"].nunique() > 1).sum()))
print("largest group:", size.max(), "copies of the same picture")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("empty cells in the table:", int(df.isnull().sum().sum()))
print("files that could not be read: 0 (all", f"{len(df):,}", "opened)")
blank = df["contrast"] < 5
print("blank pictures (almost one colour, pixel standard deviation below 5):", int(blank.sum()))
non_face = pd.read_csv("data/non_face_images.csv")
print("pictures that are not faces (icons, warning signs, 'image removed' text), checked by eye:", len(non_face))

In [ ]:
# Visualizing the missing values
# In an image dataset a "missing value" is a picture with no usable face. These are examples of both kinds.
show = list(df.index[blank][:6]) + list(df.index[df["file"].isin(non_face["file"])][:18])
fig, axes = plt.subplots(3, 8, figsize=(14, 5.8))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X[i], cmap="gray", vmin=0, vmax=255)
    ax.set_title(("blank" if blank[i] else "not a face") + f"\n({df.label[i]})", fontsize=8)
    ax.axis("off")
plt.suptitle("Images without a usable face: 6 of the blank ones and 18 of the 58 non-face ones", y=1.0)
save("data_missing_faces.png")

### What did you know about your dataset?

Answer Here.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print(list(df.columns))

In [ ]:
# Dataset Describe
STAT_COLS = ["brightness", "contrast", "sharpness", "entropy", "edge_eyes", "edge_nose", "edge_mouth", "asymmetry", "flat_share"]
df[STAT_COLS].describe().round(2)

### Variables Description 

Answer Here.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
display(df[["file", "folder", "label", "width", "height", "mode", "hash"]].nunique().rename("unique values").to_frame())
print(df["label"].value_counts())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# 1) How the non-face pictures were found: icons and "image removed" banners are mostly one flat colour,
#    so I ranked all pictures by the share of pixels close to their most common grey value and looked
#    at the 250 flattest ones by eye. The 58 that are not faces are listed in data/non_face_images.csv.
cand = df[df["contrast"] >= 5].sort_values("flat_share", ascending=False).head(60)
flagged = set(non_face["file"])
fig, axes = plt.subplots(4, 15, figsize=(16, 4.8))
for ax, (i, row) in zip(axes.ravel(), cand.iterrows()):
    ax.imshow(X[i], cmap="gray", vmin=0, vmax=255)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color("red" if row["file"] in flagged else "white"); sp.set_linewidth(2.5)
plt.suptitle("The 60 flattest pictures (red frame = removed as not a face)", y=0.98)
save("wrangling_flat_candidates.png")

# 2) Apply the cleaning rules in order (see deepfer.clean_dataset)
df = deepfer.clean_dataset(df, non_face["file"])
df["status"] = df["drop_reason"].replace("", "kept")
display(pd.crosstab(df["status"], df["folder"], margins=True, margins_name="total"))

# 3) The clean data, the integer label of every picture and the three splits.
#    The validation folder is kept apart as the TEST set (never used for any choice). From the train
#    folder 15% goes to a VALIDATION set for early stopping and model choice, stratified by emotion.
LABEL_ID = {c: i for i, c in enumerate(CLASSES)}
Y = df["label"].map(LABEL_ID).to_numpy()
clean = df[df["drop_reason"] == ""].copy()
te_idx = clean.index[clean["folder"] == "validation"].to_numpy()
pool = clean[clean["folder"] == "train"]
tr_idx, va_idx = train_test_split(pool.index.to_numpy(), test_size=0.15, stratify=pool["label"], random_state=SEED)
clean["split"] = "train"
clean.loc[va_idx, "split"] = "validation"
clean.loc[te_idx, "split"] = "test"
print(f"clean pictures: {len(clean):,} of {len(df):,} ({len(clean) / len(df):.1%})")
print(f"train {len(tr_idx):,} | validation {len(va_idx):,} | test {len(te_idx):,}")

### What all manipulations have you done and insights you found?

Answer Here.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 - How many faces per emotion?

In [ ]:
# Chart - 1 visualization code
# Univariate: how many clean pictures each emotion has, in the train folder and in the test folder
cnt = pd.crosstab(clean["label"], clean["folder"]).rename(columns={"train": "train folder", "validation": "test folder"})
share = cnt / cnt.sum() * 100
fig, ax = plt.subplots(1, 2, figsize=(14, 4.3))
cnt.plot.bar(ax=ax[0], color=["#4C72B0", "#DD8452"], rot=0)
for c in ax[0].containers:
    ax[0].bar_label(c, fontsize=8)
ax[0].set_title("Clean pictures per emotion")
ax[0].set_xlabel("")
share.plot.bar(ax=ax[1], color=["#4C72B0", "#DD8452"], rot=0)
for c in ax[1].containers:
    ax[1].bar_label(c, fmt="%.1f", fontsize=8)
ax[1].set_title("Share of each emotion (%)")
ax[1].set_xlabel("")
save("chart01_class_balance.png")
print(share.round(1))
print("largest / smallest class:", round(cnt["train folder"].max() / cnt["train folder"].min(), 1))

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 2 - What the faces look like

In [ ]:
# Chart - 2 visualization code
# Univariate (the images themselves): eight random clean faces of every emotion
rs = np.random.RandomState(1)
fig, axes = plt.subplots(7, 8, figsize=(10.5, 9.8))
for r, c in enumerate(CLASSES):
    for k, i in enumerate(rs.choice(clean.index[clean["label"] == c], 8, replace=False)):
        axes[r, k].imshow(X[i], cmap="gray", vmin=0, vmax=255)
        axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
    axes[r, 0].set_ylabel(c, fontsize=12, rotation=0, ha="right", va="center")
plt.suptitle("Random faces per emotion", y=0.92)
save("chart02_sample_faces.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 3 - The average face of each emotion

In [ ]:
# Chart - 3 visualization code
# Multivariate: the average face of every emotion (all 2,304 pixels averaged per class)
mean_face = {c: X[clean.index[clean["label"] == c]].mean(0) for c in CLASSES}
overall = X[clean.index].mean(0)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for ax, c in zip(axes, CLASSES + ["all"]):
    ax.imshow(overall if c == "all" else mean_face[c], cmap="gray")
    ax.set_title(c)
    ax.axis("off")
plt.suptitle("Average face per emotion", y=1.05)
save("chart03_average_faces.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 4 - Where each emotion differs from the average face

In [ ]:
# Chart - 4 visualization code
# Bivariate (pixel value vs emotion): where each emotion's average face is brighter (red) or darker (blue)
# than the average face of all emotions
diff = {c: mean_face[c] - overall for c in CLASSES}
lim = max(np.abs(d).max() for d in diff.values())
fig, axes = plt.subplots(1, 7, figsize=(16, 2.8))
for ax, c in zip(axes, CLASSES):
    im = ax.imshow(diff[c], cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(c)
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.012, label="grey level difference")
plt.suptitle("Average face of each emotion minus the overall average face", y=1.05)
save("chart04_face_difference_maps.png")
for c in CLASSES:
    d = diff[c]
    print(f"{c:9s} eyes band {d[10:24].mean():+5.1f}   mouth band {d[31:45, 10:38].mean():+5.1f}")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 5 - Brightness by emotion

In [ ]:
# Chart - 5 visualization code
# Numerical vs categorical: brightness of the picture by emotion
order = CLASSES
fig, ax = plt.subplots(figsize=(11, 4.3))
sns.boxplot(data=clean, x="label", y="brightness", order=order, palette="Set2", fliersize=1.5, ax=ax)
ax.set_title("Average grey level of the picture by emotion (0 = black, 255 = white)")
ax.set_xlabel("")
save("chart05_brightness_by_emotion.png")
print(clean.groupby("label")["brightness"].median().round(1).sort_values())

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 6 - Contrast and sharpness by emotion

In [ ]:
# Chart - 6 visualization code
# Numerical vs categorical: contrast and sharpness by emotion
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
sns.boxplot(data=clean, x="label", y="contrast", order=order, palette="Set2", fliersize=1.5, ax=ax[0])
ax[0].set_title("Contrast (standard deviation of the grey levels)")
sns.boxplot(data=clean.assign(sharp=np.log10(clean["sharpness"] + 1)), x="label", y="sharp", order=order,
            palette="Set2", fliersize=1.5, ax=ax[1])
ax[1].set_title("Sharpness (log10 of the Laplacian variance)")
for a in ax:
    a.set_xlabel("")
    a.tick_params(axis="x", rotation=30)
save("chart06_contrast_sharpness.png")
print((clean["sharpness"] < clean["sharpness"].quantile(0.05)).groupby(clean["label"]).mean().round(3).rename("share of blurriest 5%"))

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 7 - Edge strength in the eye, nose and mouth regions

In [ ]:
# Chart - 7 visualization code
# Bivariate: how strong the edges are in the eye, nose and mouth bands of the face, per emotion
reg = clean.groupby("label")[["edge_eyes", "edge_nose", "edge_mouth"]].mean().loc[CLASSES]
z = (reg - reg.mean()) / reg.std()
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.heatmap(z.T, annot=reg.T.round(1), fmt="", cmap="coolwarm", center=0, cbar_kws={"label": "z-score across emotions"}, ax=ax)
ax.set_yticklabels(["eyes and brows", "nose", "mouth"], rotation=0)
ax.set_title("Mean edge strength by face region (numbers) and how it ranks (colour)")
ax.set_xlabel("")
save("chart07_edges_by_region.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 8 - Removed pictures by reason and emotion

In [ ]:
# Chart - 8 visualization code
# Categorical vs categorical: the pictures that were removed, by reason and emotion
rem = pd.crosstab(df.loc[df["drop_reason"] != "", "label"], df.loc[df["drop_reason"] != "", "drop_reason"]).loc[CLASSES]
rate = (df["drop_reason"] != "").groupby(df["label"]).mean().loc[CLASSES] * 100
fig, ax = plt.subplots(1, 2, figsize=(15, 4.3))
rem.plot.bar(stacked=True, ax=ax[0], colormap="tab10", rot=0)
ax[0].set_title("Removed pictures by reason")
ax[0].legend(fontsize=8)
ax[0].set_xlabel("")
rate.plot.bar(ax=ax[1], color="#C44E52", rot=0)
ax[1].bar_label(ax[1].containers[0], fmt="%.1f%%", fontsize=8)
ax[1].set_title("Share of each emotion that was removed (%)")
ax[1].set_xlabel("")
save("chart08_removed_by_reason.png")
print(rem.sum().sort_values(ascending=False))

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 9 - Same picture, different labels

In [ ]:
# Chart - 9 visualization code
# Label noise: the same picture saved under two or more emotions
conf = df[df["drop_reason"] == "same image, different labels"]
groups = [g for _, g in conf.groupby("hash")][:12]
fig, axes = plt.subplots(2, 6, figsize=(14, 5.4))
for ax, g in zip(axes.ravel(), groups):
    ax.imshow(X[g.index[0]], cmap="gray", vmin=0, vmax=255)
    labels = " / ".join(sorted(g["label"].unique()))
    ax.set_title(labels, fontsize=9)
    ax.axis("off")
plt.suptitle("Same picture, different emotion labels (all copies removed)", y=1.0)
save("chart09_conflicting_labels.png")
pairs = conf.groupby("hash")["label"].apply(lambda s: " / ".join(sorted(s.unique()))).value_counts()
print(pairs.head(8))

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 10 - What HOG features see

In [ ]:
# Chart - 10 visualization code
# What the handcrafted features see: one face per emotion and its HOG picture (edge directions per 8x8 cell)
rs = np.random.RandomState(3)
fig, axes = plt.subplots(2, 7, figsize=(15, 4.8))
for k, c in enumerate(CLASSES):
    i = rs.choice(clean.index[clean["label"] == c])
    _, hog_img = hog(X[i], orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    axes[0, k].imshow(X[i], cmap="gray"); axes[0, k].set_title(c); axes[0, k].axis("off")
    axes[1, k].imshow(hog_img, cmap="magma"); axes[1, k].axis("off")
plt.suptitle("Faces (top) and their HOG edge pictures (bottom)", y=1.0)
save("chart10_hog_examples.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 11 - Eigenfaces (pixel PCA)

In [ ]:
# Chart - 11 visualization code
# Eigenfaces: PCA on the raw pixels of the training rows. The first components are the main ways faces vary.
pix = X[tr_idx].reshape(len(tr_idx), -1).astype(np.float32) / 255
pca_pix = PCA(n_components=200, random_state=SEED).fit(pix)
cum = np.cumsum(pca_pix.explained_variance_ratio_)
fig = plt.figure(figsize=(15, 4.2))
for k in range(10):
    ax = fig.add_subplot(2, 7, k + 1 + (k // 5) * 2)
    ax.imshow(pca_pix.components_[k].reshape(48, 48), cmap="RdBu_r")
    ax.set_title(f"PC{k + 1} ({pca_pix.explained_variance_ratio_[k]:.1%})", fontsize=9)
    ax.axis("off")
ax = fig.add_subplot(1, 7, (6, 7))
ax.plot(np.arange(1, 201), cum * 100)
for t in (80, 90, 95):
    n = int(np.searchsorted(cum, t / 100) + 1)
    ax.axhline(t, ls=":", c="grey"); ax.text(n + 3, t - 4, f"{n} PCs for {t}%", fontsize=8)
ax.set_xlabel("number of components"); ax.set_ylabel("variance explained (%)")
plt.suptitle("The first 10 eigenfaces and the variance explained by the pixel PCA", y=1.02)
save("chart11_eigenfaces.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 12 - t-SNE map of the handcrafted features

In [ ]:
# Chart - 12 visualization code
# Multivariate: a 2D map (t-SNE) of the HOG + LBP features of 3,000 training faces, coloured by emotion
rs = np.random.RandomState(SEED)
sample = rs.choice(tr_idx, 1000 if QUICK else 3000, replace=False)
feat = deepfer.hog_lbp_features(X[sample])
emb = TSNE(n_components=2, perplexity=40, init="pca", random_state=SEED).fit_transform(
    PCA(50, random_state=SEED).fit_transform(StandardScaler().fit_transform(feat)))
fig, ax = plt.subplots(figsize=(8.5, 7))
pal = dict(zip(CLASSES, sns.color_palette("tab10", 7)))
for c in CLASSES:
    m = Y[sample] == LABEL_ID[c]
    ax.scatter(emb[m, 0], emb[m, 1], s=7, alpha=0.6, color=pal[c], label=c)
ax.legend(markerscale=3)
ax.set_title("t-SNE map of handcrafted features: the emotions overlap heavily")
ax.set_xticks([]); ax.set_yticks([])
save("chart12_tsne_hog.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 13 - Lighting conditions

In [ ]:
# Chart - 13 visualization code
# Lighting conditions: brightness against contrast for all clean pictures, with examples from dark to bright
fig = plt.figure(figsize=(15, 5))
ax = fig.add_subplot(1, 2, 1)
hb = ax.hexbin(clean["brightness"], clean["contrast"], gridsize=45, cmap="viridis", mincnt=1, bins="log")
fig.colorbar(hb, ax=ax, label="pictures (log)")
ax.axvline(70, c="w", ls="--"); ax.axvline(190, c="w", ls="--")
ax.set_xlabel("brightness"); ax.set_ylabel("contrast")
ax.set_title("Lighting of the pictures (dashed: dark < 70, bright > 190)")
q = clean["brightness"].quantile(np.linspace(0.005, 0.995, 8)).to_numpy()
for k, b in enumerate(q):
    i = (clean["brightness"] - b).abs().idxmin()
    a = fig.add_subplot(2, 8, 5 + k + (k // 4) * 4)
    a.imshow(X[i], cmap="gray", vmin=0, vmax=255); a.set_title(f"{clean.brightness[i]:.0f}", fontsize=9); a.axis("off")
save("chart13_lighting.png")
print(f"dark (<70): {(clean.brightness < 70).mean():.1%}   bright (>190): {(clean.brightness > 190).mean():.1%}   "
      f"low contrast (<25): {(clean.contrast < 25).mean():.1%}")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
# Left: correlation between the image statistics. Right: how similar the average faces of the emotions are
# (Pearson correlation of the 2,304 pixels of the average faces after removing the overall average face).
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
corr = clean[STAT_COLS[:-1]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[0], cbar=False)
ax[0].set_title("Correlation between image statistics")
sim = pd.DataFrame(np.corrcoef(np.stack([diff[c].ravel() for c in CLASSES])), index=CLASSES, columns=CLASSES)
sns.heatmap(sim, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[1], cbar=False)
ax[1].set_title("Similarity of the emotions' average faces")
save("chart14_correlation.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

#### Chart - 15 - Pair Plot 

In [ ]:
# Pair Plot visualization code
rs = np.random.RandomState(SEED)
pp = clean.loc[rs.choice(clean.index, 1500, replace=False), ["brightness", "contrast", "edge_eyes", "edge_mouth", "label"]]
g = sns.pairplot(pp, hue="label", hue_order=CLASSES, palette="tab10", plot_kws={"s": 8, "alpha": 0.5}, diag_kind="kde", height=2.3)
g.fig.suptitle("Pair plot of four image statistics (1,500 random faces)", y=1.02)
save("chart15_pairplot.png")

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Answer Here.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
groups = [clean.loc[clean["label"] == c, "brightness"] for c in CLASSES]
h, p = stats.kruskal(*groups)
eps2 = (h - len(CLASSES) + 1) / (len(clean) - len(CLASSES))        # epsilon squared effect size
print(f"Kruskal-Wallis H = {h:.1f}, p-value = {p:.2e}, effect size (epsilon squared) = {eps2:.4f}")
print(clean.groupby("label")["brightness"].median().round(1).to_string())
print("reject H0" if p < 0.05 else "cannot reject H0", "at the 5% level")

##### Which statistical test have you done to obtain P-Value?

Answer Here.

##### Why did you choose the specific statistical test?

Answer Here.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
tab = pd.crosstab(clean["label"], clean["folder"])
chi2, p, dof, expected = stats.chi2_contingency(tab)
cramer_v = np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1)))
print(f"chi-square = {chi2:.2f}, degrees of freedom = {dof}, p-value = {p:.3f}, Cramer's V = {cramer_v:.3f}")
print((tab / tab.sum() * 100).round(2))
print("reject H0" if p < 0.05 else "cannot reject H0", "at the 5% level")

##### Which statistical test have you done to obtain P-Value?

Answer Here.

##### Why did you choose the specific statistical test?

Answer Here.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
happy = clean.loc[clean["label"] == "happy", "edge_mouth"]
neutral = clean.loc[clean["label"] == "neutral", "edge_mouth"]
u, p = stats.mannwhitneyu(happy, neutral, alternative="greater")
cles = u / (len(happy) * len(neutral))       # chance that a random happy face has stronger mouth edges than a random neutral one
print(f"Mann-Whitney U = {u:,.0f}, one-sided p-value = {p:.2e}")
print(f"median mouth edge strength: happy {happy.median():.2f} vs neutral {neutral.median():.2f}; "
      f"P(happy > neutral) = {cles:.3f}")
print("reject H0" if p < 0.05 else "cannot reject H0", "at the 5% level")

##### Which statistical test have you done to obtain P-Value?

Answer Here.

##### Why did you choose the specific statistical test?

Answer Here.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
removed = df["drop_reason"].isin(["blank image", "not a face"])
print(f"pictures without a usable face that were removed: {removed.sum()} "
      f"({(df.drop_reason == 'blank image').sum()} blank, {(df.drop_reason == 'not a face').sum()} not a face)")
print("missing labels left in the clean data:", int(clean["label"].isnull().sum()), "| pixels are uint8, so they can't be missing")
print("every clean picture is 48x48 grayscale:", bool((clean[["width", "height"]] == 48).all().all() and (clean["mode"] == "L").all()))

#### What all missing value imputation techniques have you used and why did you use those techniques?

Answer Here.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Outliers in a face dataset are pictures with extreme lighting, very low contrast or blur.
z = clean[["brightness", "contrast"]].apply(lambda s: (s - s.mean()) / s.std())
z["sharpness"] = np.log10(clean["sharpness"] + 1)
z["sharpness"] = (z["sharpness"] - z["sharpness"].mean()) / z["sharpness"].std()
out = (z.abs() > 3)
print("pictures beyond 3 standard deviations:", out.sum().to_dict(), "| any:", int(out.any(axis=1).sum()),
      f"({out.any(axis=1).mean():.2%})")
fig, axes = plt.subplots(3, 8, figsize=(14, 5.6))
for r, (col, asc, name) in enumerate([("brightness", True, "darkest"), ("contrast", True, "lowest contrast"),
                                      ("sharpness", True, "blurriest")]):
    for k, i in enumerate(clean.sort_values(col, ascending=asc).index[:8]):
        axes[r, k].imshow(X[i], cmap="gray", vmin=0, vmax=255)
        axes[r, k].set_title(clean.label[i], fontsize=8)
        axes[r, k].axis("off")
    axes[r, 0].text(-6, 24, name, ha="right", va="center", fontsize=11)
plt.suptitle("The most extreme clean pictures: real faces, so they are kept", y=1.0)
save("prep_outliers_kept.png")

##### What all outlier treatment techniques have you used and why did you use those techniques?

Answer Here.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# The only categorical column is the target (the emotion). It becomes an integer 0-6, which is what
# PyTorch's cross-entropy loss and scikit-learn expect. No one-hot columns are needed.
print(LABEL_ID)
print("example:", df.loc[clean.index[:5], ["file", "label"]].assign(y=Y[clean.index[:5]]).to_string(index=False))

#### What all categorical encoding techniques have you used & why did you use those techniques?

Answer Here.

### 4. Textual Data Preprocessing 
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

This dataset has no text, only face images, so the ten text steps below don't apply. The image pre-processing (cleaning, scaling, resizing, augmentation) is in steps 1-3 above and 5-9 below.

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 2. Lower Casing

In [ ]:
# Lower Casing
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

In [ ]:
# Remove White spaces
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 7. Tokenization

In [ ]:
# Tokenization
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

##### Which text normalization technique have you used and why?

Not applicable: there is no text in this dataset.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
# not needed here: the data are images, there is no text column
print("not applicable: the dataset has no text, only face images")

##### Which text vectorization technique have you used and why?

Not applicable: there is no text in this dataset.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# Two kinds of features are used in this project:
#  a) for the CNNs: the 2,304 raw pixels. The networks learn their own features, and new variations of every
#     training face are created on the fly by data augmentation (section 5 below).
#  b) for the classic baseline: handcrafted features, HOG (edge directions in 8x8 cells, 900 numbers) and
#     LBP (texture codes counted in a 4x4 grid of regions, 160 numbers).
t0 = time.time()
H_raw = deepfer.hog_lbp_features(X)                    # all pictures, so the arrays line up with X and Y
H_eq = deepfer.hog_lbp_features(X, use_clahe=True)     # the same after CLAHE histogram equalisation
print(f"HOG + LBP features: {H_raw.shape[1]} per picture, computed in {time.time() - t0:.0f}s")

# Does CLAHE (evening out the lighting) make the handcrafted features better? Quick check on validation.
quick_tr = tr_idx[:4000] if QUICK else tr_idx
check = {}
for name, H in [("raw pictures", H_raw), ("after CLAHE", H_eq)]:
    m = make_pipeline(StandardScaler(), PCA(150, random_state=SEED), LogisticRegression(max_iter=1000))
    m.fit(H[quick_tr], Y[quick_tr])
    p = m.predict(H[va_idx])
    check[name] = {"val accuracy": accuracy_score(Y[va_idx], p), "val macro F1": f1_score(Y[va_idx], p, average="macro")}
check = pd.DataFrame(check).T.round(4)
display(check)
USE_CLAHE = check.loc["after CLAHE", "val macro F1"] > check.loc["raw pictures", "val macro F1"]
H = H_eq if USE_CLAHE else H_raw
print("handcrafted features use", "CLAHE pictures" if USE_CLAHE else "raw pictures")

# the features are strongly correlated (neighbouring HOG cells overlap), which PCA handles in step 7
c = np.corrcoef(H[tr_idx[:3000]].T)
print(f"share of feature pairs with |correlation| > 0.5: {(np.abs(c[np.triu_indices_from(c, 1)]) > 0.5).mean():.2%}")

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# ANOVA F-test on the training rows only: how strongly each pixel (and each HOG cell) differs between the 7 emotions
pix_tr = X[tr_idx].reshape(len(tr_idx), -1).astype(np.float32)
f_pix, _ = f_classif(pix_tr, Y[tr_idx])
f_map = f_pix.reshape(48, 48)
f_hog, _ = f_classif(H[tr_idx][:, :900], Y[tr_idx])
# HOG: 5x5 blocks of 2x2 cells x 9 directions -> average F per block, shown on the face grid
f_block = f_hog.reshape(5, 5, 36).mean(2)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
ax[0].imshow(overall, cmap="gray"); ax[0].set_title("average face"); ax[0].axis("off")
ax[1].imshow(overall, cmap="gray"); im = ax[1].imshow(f_map, cmap="inferno", alpha=0.65)
ax[1].set_title("F-score of each pixel"); ax[1].axis("off"); fig.colorbar(im, ax=ax[1], fraction=0.046)
ax[2].imshow(overall, cmap="gray", extent=(0, 48, 48, 0))
im = ax[2].imshow(f_block, cmap="inferno", alpha=0.65, extent=(4, 44, 44, 4))
ax[2].set_title("F-score of the HOG blocks"); ax[2].axis("off"); fig.colorbar(im, ax=ax[2], fraction=0.046)
plt.suptitle("Which parts of the face separate the emotions (ANOVA F-test, training rows)", y=1.02)
save("prep_feature_importance_f_test.png")

regions = {"forehead (rows 0-9)": f_map[:10], "eyes and brows (10-23)": f_map[10:24], "nose (24-30)": f_map[24:31],
           "mouth and chin (31-47)": f_map[31:]}
print(pd.Series({k: v.mean() for k, v in regions.items()}).round(1).rename("mean F-score").to_string())
print(f"\npixels with F > 50: {(f_pix > 50).mean():.1%} | the weakest 25% of pixels have F below {np.percentile(f_pix, 25):.1f}")

##### What all feature selection methods have you used  and why?

Answer Here.

##### Which all features you found important and why?

Answer Here.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
# 1) Pixels 0-255 -> 0-1 and standardised. This happens inside every model (deepfer.InputAdapter), so the
#    saved models take a plain 48x48 grey face and the app can't forget a step.
PIXEL_MEAN = float(X[tr_idx].mean() / 255)
PIXEL_STD = float(X[tr_idx].std() / 255)
print(f"training pixels: mean {PIXEL_MEAN:.4f}, standard deviation {PIXEL_STD:.4f}")
# 2) For the ImageNet models: resize 48 -> 128 (bilinear), copy grey to 3 channels, ImageNet mean and std.
# 3) Data augmentation, for training only, on the GPU for a whole batch at once: mirror, rotation +-15 deg,
#    zoom +-12%, shift +-10%, brightness +-20%, contrast +-30% and a random grey patch (cut-out).
rs = np.random.RandomState(7)
picks = [rs.choice(clean.index[(clean["label"] == c) & (clean["split"] == "train")]) for c in ["happy", "fear", "surprise", "sad"]]
torch.manual_seed(0)
fig, axes = plt.subplots(4, 9, figsize=(14, 6.6))
for r, i in enumerate(picks):
    base = torch.from_numpy(X[i]).float().div(255).view(1, 1, 48, 48).repeat(8, 1, 1, 1)
    aug = deepfer.augment(base).numpy()[:, 0]
    axes[r, 0].imshow(X[i], cmap="gray", vmin=0, vmax=255); axes[r, 0].set_title("original" if r == 0 else "", fontsize=9)
    for k in range(8):
        axes[r, k + 1].imshow(aug[k], cmap="gray", vmin=0, vmax=1)
        if r == 0:
            axes[r, k + 1].set_title(f"augmented {k + 1}", fontsize=9)
    for a in axes[r]:
        a.axis("off")
plt.suptitle("Data augmentation: every epoch the network sees a new random version of each face", y=1.0)
save("prep_augmentation.png")

# what the pre-trained models receive: the same face at 128x128 in 3 channels
adapter = deepfer.InputAdapter(128, True, deepfer.IMAGENET_MEAN, deepfer.IMAGENET_STD)
x128 = adapter(torch.from_numpy(X[picks[0]]).float().div(255).view(1, 1, 48, 48))
print("input of the ImageNet models:", tuple(x128.shape), f"mean {x128.mean():.2f}, std {x128.std():.2f}")

Answer Here.

### 6. Data Scaling

In [ ]:
# Scaling your data
# Handcrafted features: StandardScaler (mean 0, std 1 per feature), fitted on the training rows inside the model pipeline.
sc = StandardScaler().fit(H[tr_idx])
print("HOG+LBP after scaling, training rows: mean", round(float(sc.transform(H[tr_idx]).mean()), 4),
      "std", round(float(sc.transform(H[tr_idx]).std()), 4))
# Pixels: standardised with the training mean and std inside the custom CNN.
batch = deepfer.InputAdapter(48, False, [PIXEL_MEAN], [PIXEL_STD])(torch.from_numpy(X[tr_idx[:2000]]).float().div(255).unsqueeze(1))
print(f"pixels after scaling, 2,000 training faces: mean {batch.mean():.3f}, std {batch.std():.3f}")

##### Which method have you used to scale you data and why?

Answer Here.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Answer Here.

In [ ]:
# DImensionality Reduction (If needed)
# PCA on the scaled HOG + LBP features of the training rows (for the classic baseline only)
pca_h = PCA(random_state=SEED).fit(sc.transform(H[tr_idx]))
cum_h = np.cumsum(pca_h.explained_variance_ratio_)
need = {t: int(np.searchsorted(cum_h, t) + 1) for t in (0.80, 0.90, 0.95, 0.99)}
print("components needed:", {f"{int(k * 100)}%": v for k, v in need.items()}, "of", H.shape[1])
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(np.arange(1, len(cum_h) + 1), cum_h * 100)
for t, n in need.items():
    ax.axvline(n, ls=":", c="grey"); ax.text(n + 8, t * 100 - 6, f"{n}", fontsize=8)
ax.set_xlabel("PCA components"); ax.set_ylabel("variance explained (%)")
ax.set_title("PCA of the 1,060 handcrafted features")
save("prep_pca_handcrafted.png")

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# (the split itself was made in section 3, right after cleaning, so no later step could look at the test set)
tab = pd.crosstab(clean["label"], clean["split"])[["train", "validation", "test"]]
tab.loc["total"] = tab.sum()
display(tab)
print((tab.loc["total"] / tab.loc["total"].sum()).round(3).to_dict())
assert not set(df.loc[tr_idx, "hash"]) & set(df.loc[te_idx, "hash"]), "a training picture is also in the test set"
assert not set(df.loc[va_idx, "hash"]) & set(df.loc[te_idx, "hash"])
print("no picture is shared between train/validation and test")

##### What data splitting ratio have you used and why? 

Answer Here.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Answer Here.

In [ ]:
# Handling Imbalanced Dataset (If needed)
counts = pd.Series(Y[tr_idx]).value_counts().sort_index()
counts.index = CLASSES
w_balanced = (len(tr_idx) / (len(CLASSES) * counts))
w_sqrt = np.sqrt(w_balanced) / np.sqrt(w_balanced).mean()      # softer weights: square root of the balanced ones
weights = pd.DataFrame({"training pictures": counts, "share %": (counts / counts.sum() * 100).round(1),
                        "balanced weight": w_balanced.round(2), "sqrt weight (used if it helps)": w_sqrt.round(2)})
display(weights)
CLASS_WEIGHTS_SQRT = w_sqrt.to_numpy().tolist()
print("largest / smallest class in training:", round(counts.max() / counts.min(), 1))

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Answer Here.

## ***7. ML Model Implementation***

#### Evaluation helpers
Every model is scored the same way: on the validation rows (used for all choices) and on the test rows (the original validation folder, used only to report). The main metrics are accuracy and macro F1, because macro F1 gives the rare emotions (disgust) the same weight as the common ones.

In [ ]:
# Helpers used by all models: scores on validation and test, kept in one table
RESULTS, PROBS = {}, {}
y_va, y_te = Y[va_idx], Y[te_idx]
fit_idx = tr_idx[:1500] if QUICK else tr_idx            # QUICK = tiny code check only
E = (lambda n: 1) if QUICK else (lambda n: n)          # number of epochs


def softmax(z):
    return torch.softmax(torch.as_tensor(z, dtype=torch.float32), 1).numpy()


def evaluate(name, p_val, p_test, show=True):
    """Keep the class probabilities of a model and its validation and test scores."""
    PROBS[name] = (np.asarray(p_val), np.asarray(p_test))
    RESULTS[name] = {**{f"val {k}": v for k, v in deepfer.scores(y_va, np.argmax(p_val, 1)).items()},
                     **{f"test {k}": v for k, v in deepfer.scores(y_te, np.argmax(p_test, 1)).items()}}
    if show:
        display(pd.DataFrame(RESULTS[name], index=[name]).round(4))


def score_chart(names, title, fname, split="test"):
    cols = ["accuracy", "macro precision", "macro recall", "macro F1"]
    t = pd.DataFrame({n: [RESULTS[n][f"{split} {c}"] for c in cols] for n in names}, index=cols)
    ax = t.plot.bar(figsize=(11, 4), rot=0, width=0.8)
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.3f", fontsize=7)
    ax.set_ylim(0, max(0.9, t.values.max() + 0.1))
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper right")
    save(fname)


def plot_history(hist, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].plot(hist["epoch"], hist["train loss"], label="train loss (augmented, smoothed labels)")
    ax[0].plot(hist["epoch"], hist["val loss"], label="validation loss")
    ax[0].legend(); ax[0].set_xlabel("epoch"); ax[0].set_title("loss")
    ax[1].plot(hist["epoch"], hist["val accuracy"], label="validation accuracy")
    ax[1].plot(hist["epoch"], hist["val macro F1"], label="validation macro F1")
    best = hist["val macro F1"].idxmax()
    ax[1].axvline(hist.loc[best, "epoch"], ls=":", c="grey", label=f"best epoch {hist.loc[best, 'epoch']}")
    ax[1].legend(); ax[1].set_xlabel("epoch"); ax[1].set_title("validation scores")
    plt.suptitle(title, y=1.02)
    save(fname)


def logits(model, Xd, tta=False):
    return deepfer.predict_logits(model, Xd, DEVICE, tta=tta).numpy()


# All pictures stay on the GPU as uint8 (about 80 MB), so a batch never waits for the CPU.
X_dev = torch.from_numpy(X).to(DEVICE)
Y_t = torch.from_numpy(Y)
rows = lambda idx: (X_dev[torch.from_numpy(idx).to(DEVICE)], Y_t[torch.from_numpy(idx)])
Xtr, ytr = rows(fit_idx)
Xva, yva = rows(va_idx)
Xte, yte = rows(te_idx)
print(f"train {len(Xtr):,} | validation {len(Xva):,} | test {len(Xte):,} pictures on {DEVICE}")

### ML Model - 1 - Classic baseline: HOG + LBP features with logistic regression

In [ ]:
# ML Model - 1 Implementation
# Classic computer vision baseline: HOG + LBP features -> scaling -> multinomial logistic regression
# (first version with default settings: all 1,060 features, C = 1, no class weights)
t0 = time.time()
lr_default = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
# Fit the Algorithm
lr_default.fit(H[fit_idx], Y[fit_idx])
print(f"trained in {time.time() - t0:.0f}s")
# Predict on the model
evaluate("HOG+LBP LR (default)", lr_default.predict_proba(H[va_idx]), lr_default.predict_proba(H[te_idx]))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
score_chart(["HOG+LBP LR (default)"], "ML Model 1 (HOG + LBP, logistic regression) on the test set", "model1_scores.png")
rep = pd.DataFrame(classification_report(y_te, PROBS["HOG+LBP LR (default)"][1].argmax(1), target_names=CLASSES,
                                         output_dict=True)).T.loc[CLASSES, ["precision", "recall", "f1-score"]]
display(rep.round(3))

Answer Here.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Grid search with 3-fold stratified cross-validation on the training rows, scored by macro F1
grid = GridSearchCV(
    make_pipeline(StandardScaler(), PCA(random_state=SEED), LogisticRegression(max_iter=3000)),
    {"pca__n_components": [0.90, 0.95],
     "logisticregression__C": [0.05, 0.5],
     "logisticregression__class_weight": [None, "balanced"]},
    cv=StratifiedKFold(3, shuffle=True, random_state=SEED), scoring="f1_macro", n_jobs=-1)

# Fit the Algorithm
t0 = time.time()
grid.fit(H[fit_idx], Y[fit_idx])
print(f"8 settings x 3 folds in {time.time() - t0:.0f}s, best: {grid.best_params_}")
cv_tab = pd.DataFrame(grid.cv_results_)
cv_tab = cv_tab[["param_pca__n_components", "param_logisticregression__C", "param_logisticregression__class_weight",
                 "mean_test_score", "std_test_score"]].sort_values("mean_test_score", ascending=False)
cv_tab.columns = ["PCA variance", "C", "class weight", "CV macro F1", "CV std"]
display(cv_tab.round(4))

# Predict on the model
evaluate("HOG+LBP LR (tuned)", grid.predict_proba(H[va_idx]), grid.predict_proba(H[te_idx]))
score_chart(["HOG+LBP LR (default)", "HOG+LBP LR (tuned)"], "ML Model 1: default vs tuned (test set)", "model1_tuned_scores.png")

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

### ML Model - 2 - DeepFER-Net: a custom CNN trained from scratch

In [ ]:
# ML Model - 2 Implementation
# DeepFER-Net: a CNN designed for 48x48 faces (code in deepfer.DeepFERNet)
#  - stem: two 3x3 convolutions at full size, so small details like eye corners are kept
#  - three residual stages 48 -> 24 -> 12 -> 6 pixels with 128 -> 256 -> 512 channels; every block is
#    conv-BN-ReLU-conv-BN + squeeze-excitation (channel attention) + a shortcut connection
#  - global average pooling -> dropout 0.4 -> 7 outputs (softmax inside the cross-entropy loss)
cnn0 = deepfer.build_model("deepfer_cnn", pixel_mean=PIXEL_MEAN, pixel_std=PIXEL_STD)

shapes = []
hooks = [m.register_forward_hook(lambda m, i, o, n=n: shapes.append((n, tuple(o.shape[1:]), sum(p.numel() for p in m.parameters()))))
         for n, m in cnn0[1].named_children()]
cnn0.eval()(torch.zeros(1, 1, 48, 48))
for h in hooks:
    h.remove()
display(pd.DataFrame(shapes, columns=["block", "output shape (channels, height, width)", "parameters"]))
print(f"total parameters: {deepfer.count_params(cnn0):,}")

# First version: the plain network, trained without augmentation and without class weights
NO_AUG = dict(rot=0, scale=0, shift=0, flip=False, bright=0, contrast=0, erase_p=0)
t0 = time.time()
cnn0, hist_cnn0 = deepfer.train_model(cnn0, Xtr, ytr, Xva, yva, DEVICE, epochs=E(30), lr=2e-3, patience=5, aug=NO_AUG)
print(f"training time {time.time() - t0:.0f}s on {DEVICE}")
evaluate("CNN (no augmentation)", softmax(logits(cnn0, Xva)), softmax(logits(cnn0, Xte)))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
plot_history(hist_cnn0, "DeepFER-Net without augmentation: learning curves", "model2_first_history.png")
score_chart(["HOG+LBP LR (tuned)", "CNN (no augmentation)"], "ML Model 2 (first version) vs the best classic model (test set)", "model2_scores.png")

Answer Here.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Short trials first: each setting gets 8 epochs (a complete one-cycle schedule) on half of the training rows,
# with augmentation on. That ranks the settings at about a sixth of the cost of full runs.
rs = np.random.RandomState(SEED)
half = rs.choice(len(fit_idx), len(fit_idx) // 2, replace=False)
Xhalf, yhalf = rows(fit_idx[half])
trials = []
for lr in [1e-3, 3e-3]:
    for cw_name, cw in [("none", None), ("sqrt", CLASS_WEIGHTS_SQRT)]:
        t0 = time.time()
        m = deepfer.build_model("deepfer_cnn", pixel_mean=PIXEL_MEAN, pixel_std=PIXEL_STD)
        m, h = deepfer.train_model(m, Xhalf, yhalf, Xva, yva, DEVICE, epochs=E(8), lr=lr, class_weights=cw,
                                   patience=99, log=lambda s: None)          # quiet: only the summary is printed
        best = h.loc[h["val macro F1"].idxmax()]
        trials.append({"max learning rate": lr, "class weights": cw_name, "val accuracy": best["val accuracy"],
                       "val macro F1": best["val macro F1"], "minutes": (time.time() - t0) / 60})
        deepfer.log_line(f"trial lr {lr} class weights {cw_name}: val macro F1 {best['val macro F1']:.4f}")
trials = pd.DataFrame(trials).sort_values("val macro F1", ascending=False)
display(trials.round(4))
BEST_LR = float(trials.iloc[0]["max learning rate"])
CW = CLASS_WEIGHTS_SQRT if trials.iloc[0]["class weights"] == "sqrt" else None
print("chosen:", BEST_LR, "class weights:", trials.iloc[0]["class weights"])

# Final training of the tuned network on all training rows: augmentation, up to 35 epochs, early stopping after 8
t0 = time.time()
cnn = deepfer.build_model("deepfer_cnn", pixel_mean=PIXEL_MEAN, pixel_std=PIXEL_STD)
cnn, hist_cnn = deepfer.train_model(cnn, Xtr, ytr, Xva, yva, DEVICE, epochs=E(35), lr=BEST_LR, class_weights=CW, patience=8)
print(f"training time {(time.time() - t0) / 60:.1f} min")
evaluate("CNN (augmented, tuned)", softmax(logits(cnn, Xva)), softmax(logits(cnn, Xte)))
deepfer.save_model(cnn, "models/deepfer_cnn.pt", "deepfer_cnn", {"size": 48, "pixel_mean": PIXEL_MEAN, "pixel_std": PIXEL_STD})
plot_history(hist_cnn, "DeepFER-Net with augmentation and the tuned settings", "model2_tuned_history.png")
score_chart(["CNN (no augmentation)", "CNN (augmented, tuned)"], "ML Model 2: first version vs augmented and tuned (test set)", "model2_tuned_scores.png")

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Answer Here.

### ML Model - 3 - Transfer learning: fine-tuned ResNet18 and EfficientNet-B0

In [ ]:
# ML Model - 3 Implementation
# Transfer learning from ImageNet with two well-known backbones: ResNet18 (11.2M parameters) and
# EfficientNet-B0 (4.0M). Every face is resized to 128x128 and copied to 3 channels inside the model, and the
# 1,000-class ImageNet head is replaced with dropout + a 7-class layer.
# Step 1 (this cell): the pre-trained body is frozen and only the new head learns ("feature extraction").
TL, TL_HIST = {}, {}
for name in ["resnet18", "efficientnet_b0"]:
    t0 = time.time()
    m = deepfer.build_model(name, size=128)
    m, h = deepfer.train_model(m, Xtr, ytr, Xva, yva, DEVICE, epochs=E(3), lr=1e-3, freeze_body=True,
                               class_weights=CW, patience=99)
    print(f"{name}: {deepfer.count_params(m):,} parameters, head-only training {time.time() - t0:.0f}s")
    evaluate(f"{name} (frozen body)", softmax(logits(m, Xva)), softmax(logits(m, Xte)))
    TL[name] = m

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
score_chart(["CNN (augmented, tuned)", "resnet18 (frozen body)", "efficientnet_b0 (frozen body)"],
            "Frozen ImageNet features vs the CNN trained on faces (test set)", "model3_frozen_scores.png")

Answer Here.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
# Step 2: fine-tune the whole network. The setting that matters most in transfer learning is how much of the
# pre-trained network is retrained, so frozen (step 1) is compared with fully fine-tuned here, for both backbones.
# The new head learns with a top rate of 1e-3 and the pre-trained body with 3e-4 (smaller, so the ImageNet
# knowledge is adjusted, not wiped out), one-cycle schedule, augmentation, early stopping after 5 epochs.
for name in ["resnet18", "efficientnet_b0"]:
    t0 = time.time()
    m, h = deepfer.train_model(TL[name], Xtr, ytr, Xva, yva, DEVICE, epochs=E(15), lr=1e-3, lr_body=3e-4,
                               class_weights=CW, patience=5)
    print(f"{name}: fine-tuning {(time.time() - t0) / 60:.1f} min")
    evaluate(f"{name} (fine-tuned)", softmax(logits(m, Xva)), softmax(logits(m, Xte)))
    TL[name], TL_HIST[name] = m, h
    deepfer.save_model(m, f"models/{name}.pt", name, {"size": 128})
    plot_history(h, f"{name} fine-tuning: learning curves", f"model3_{name}_history.png")
score_chart(["resnet18 (frozen body)", "resnet18 (fine-tuned)", "efficientnet_b0 (frozen body)", "efficientnet_b0 (fine-tuned)"],
            "ML Model 3: frozen vs fine-tuned (test set)", "model3_tuned_scores.png")

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

#### 3. Combining the models: mirrored faces and an ensemble

In [ ]:
# From here on every score comes from the saved model files. On my Mac the Apple GPU backend of PyTorch
# sometimes changed weights that sat in GPU memory after training (found by comparing them with the saved
# files), so each network is checked against its file before it is used and reloaded if anything changed.
FILES = {"CNN": "deepfer_cnn.pt", "ResNet18": "resnet18.pt", "EfficientNet-B0": "efficientnet_b0.pt"}
REF = {n: torch.load(f"models/{f}", map_location="cpu", weights_only=False)["state_dict"] for n, f in FILES.items()}
NETS = {n: deepfer.load_model(f"models/{f}", DEVICE)[0] for n, f in FILES.items()}
RELOADS = []


def net(name):
    """The network on the GPU, identical to its saved file (reloaded from the file if it changed)."""
    sd = NETS[name].state_dict()
    if any(not torch.equal(sd[k].detach().cpu(), v) for k, v in REF[name].items()):
        RELOADS.append(name)
        NETS[name] = deepfer.load_model(f"models/{FILES[name]}", DEVICE)[0]
    return NETS[name]


# the evaluation pictures stay in normal memory, every batch is copied to the GPU when it is scored
Xva_c, Xte_c = torch.from_numpy(X[va_idx]), torch.from_numpy(X[te_idx])
Z = {(n, t): (logits(net(n), Xva_c, t), logits(net(n), Xte_c, t)) for n in NETS for t in (False, True)}
combo_rows = []
for tta in (False, True):
    for k in (1, 2, 3):
        for members in itertools.combinations(NETS, k):
            zv = np.mean([Z[(n, tta)][0] for n in members], 0)
            sc = deepfer.scores(y_va, zv.argmax(1))
            combo_rows.append({"members": " + ".join(members), "n": k, "mirror TTA": tta,
                               "val accuracy": sc["accuracy"], "val macro F1": sc["macro F1"]})
combos = pd.DataFrame(combo_rows).sort_values("val macro F1", ascending=False).reset_index(drop=True)
display(combos.round(4))

# Pick the best on validation macro F1, but take a smaller combination if it is within 0.003 (cheaper, faster)
top = combos.iloc[0]["val macro F1"]
pick = combos[combos["val macro F1"] >= top - 0.003].sort_values(["n", "val macro F1"], ascending=[True, False]).iloc[0]
single_rank = combos[combos["n"] == 1].drop_duplicates("members")["members"].tolist()   # best single network first
FINAL_MEMBERS = sorted(pick["members"].split(" + "), key=single_rank.index)
FINAL_TTA = bool(pick["mirror TTA"])
FINAL_NAME = "Final: " + " + ".join(FINAL_MEMBERS) + (" with TTA" if FINAL_TTA else "")
print("chosen:", FINAL_NAME)


def final_logits(Xd):
    return np.mean([logits(net(n), Xd, FINAL_TTA) for n in FINAL_MEMBERS], 0)


zf_va, zf_te = final_logits(Xva_c), final_logits(Xte_c)
evaluate(FINAL_NAME, softmax(zf_va), softmax(zf_te))
best_single = single_rank[0]
evaluate(f"{best_single} (single, TTA)", softmax(Z[(best_single, True)][0]), softmax(Z[(best_single, True)][1]), show=False)

Answer Here.

#### 4. The final model on the test set

In [ ]:
# The final model on the test set: every emotion, the confusion matrix and how sure we can be of the numbers
pred_te = zf_te.argmax(1)
rep = pd.DataFrame(classification_report(y_te, pred_te, target_names=CLASSES, output_dict=True)).T
display(rep.round(3))
rep.round(4).to_csv("outputs/per_emotion_report.csv")

cm = confusion_matrix(y_te, pred_te)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(15, 5.6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0], cbar=False)
ax[0].set_title("Counts"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[1], cbar=False)
ax[1].set_title("Share of each true emotion (rows add up to 1)"); ax[1].set_xlabel("predicted"); ax[1].set_ylabel("true")
plt.suptitle(f"Confusion matrix of the final model on {len(y_te):,} test faces", y=1.0)
save("final_confusion_matrix.png")

# 95% bootstrap confidence intervals (1,000 resamples of the test set)
rs = np.random.RandomState(SEED)
boot = []
for _ in range(200 if QUICK else 1000):
    s = rs.randint(0, len(y_te), len(y_te))
    boot.append((accuracy_score(y_te[s], pred_te[s]), f1_score(y_te[s], pred_te[s], average="macro")))
boot = np.array(boot)
print(f"test accuracy {accuracy_score(y_te, pred_te):.4f} (95% CI {np.percentile(boot[:, 0], 2.5):.4f}-{np.percentile(boot[:, 0], 97.5):.4f})")
print(f"test macro F1 {f1_score(y_te, pred_te, average='macro'):.4f} (95% CI {np.percentile(boot[:, 1], 2.5):.4f}-{np.percentile(boot[:, 1], 97.5):.4f})")

# McNemar test: is the final model really better than another model on the same test faces?
def mcnemar(pred_a, pred_b):
    a_ok, b_ok = pred_a == y_te, pred_b == y_te
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return b, c, (stats.binomtest(b, b + c, 0.5).pvalue if b + c else 1.0)
for other in [f"{best_single} (single, TTA)", "CNN (augmented, tuned)", "HOG+LBP LR (tuned)"]:
    b, c, p = mcnemar(pred_te, PROBS[other][1].argmax(1))
    print(f"vs {other}: final right & other wrong {b}, the opposite {c}, McNemar p-value {p:.2e}")

Answer Here.

#### 5. Where does it go wrong? Error analysis

In [ ]:
# Where does it go wrong? The most common mix-ups, confident mistakes, and accuracy against confidence
pairs = [(CLASSES[i], CLASSES[j], cm[i, j], cmn[i, j]) for i in range(7) for j in range(7) if i != j]
pairs = pd.DataFrame(pairs, columns=["true", "predicted", "faces", "share of true class"]).sort_values("faces", ascending=False)
display(pairs.head(8).round(3))

p_te = softmax(zf_te)
conf = p_te.max(1)
wrong = np.where(pred_te != y_te)[0]
worst = wrong[np.argsort(-conf[wrong])][:16]
fig, axes = plt.subplots(2, 8, figsize=(15, 4.6))
for ax, k in zip(axes.ravel(), worst):
    ax.imshow(X[te_idx[k]], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"true {CLASSES[y_te[k]]}\npred {CLASSES[pred_te[k]]} {conf[k]:.0%}", fontsize=8)
    ax.axis("off")
plt.suptitle("The 16 most confident mistakes on the test set", y=1.02)
save("final_confident_mistakes.png")

# selective prediction: keep only the faces the model is most sure about
order_c = np.argsort(-conf)
cov = np.linspace(0.1, 1.0, 19)
acc_at = [(pred_te[order_c[:int(c * len(order_c))]] == y_te[order_c[:int(c * len(order_c))]]).mean() for c in cov]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(cov * 100, np.array(acc_at) * 100, marker="o")
ax.set_xlabel("share of faces answered (most confident first, %)"); ax.set_ylabel("accuracy on those faces (%)")
ax.set_title("Accuracy rises when the model may say 'not sure'")
save("final_coverage_accuracy.png")
for c in (0.5, 0.7, 0.9):
    n = int(c * len(order_c))
    print(f"answer the {c:.0%} most confident faces: accuracy {(pred_te[order_c[:n]] == y_te[order_c[:n]]).mean():.3f}, "
          f"confidence cut-off {conf[order_c[n - 1]]:.2f}")

Answer Here.

#### 6. Can the probabilities be trusted? Calibration

In [ ]:
# Can the probabilities be trusted? Temperature scaling, fitted on the validation set
T = deepfer.fit_temperature(torch.tensor(zf_va), torch.tensor(y_va))


def ece(p, y, bins=15):
    """Expected calibration error: the average gap between confidence and real accuracy."""
    c, ok = p.max(1), p.argmax(1) == y
    edges = np.linspace(0, 1, bins + 1)
    return sum(abs(ok[m].mean() - c[m].mean()) * m.mean()
               for lo, hi in zip(edges[:-1], edges[1:]) if (m := (c > lo) & (c <= hi)).any())


p_raw, p_cal = softmax(zf_te), softmax(zf_te / T)
print(f"temperature T = {T:.2f}")
print(f"test ECE before {ece(p_raw, y_te):.4f}, after {ece(p_cal, y_te):.4f}")
print(f"test log loss before {F.cross_entropy(torch.tensor(zf_te), torch.tensor(y_te)).item():.4f}, "
      f"after {F.cross_entropy(torch.tensor(zf_te / T), torch.tensor(y_te)).item():.4f}")
fig, ax = plt.subplots(figsize=(5.8, 5))
for p, lab in [(p_raw, "before"), (p_cal, f"after (T = {T:.2f})")]:
    c, ok = p.max(1), p.argmax(1) == y_te
    edges = np.linspace(0, 1, 11)
    mids = [(c[(c > lo) & (c <= hi)].mean(), ok[(c > lo) & (c <= hi)].mean()) for lo, hi in zip(edges[:-1], edges[1:]) if ((c > lo) & (c <= hi)).sum() > 20]
    ax.plot(*zip(*mids), marker="o", label=lab)
ax.plot([0, 1], [0, 1], ls=":", c="grey", label="perfect")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.legend(); ax.set_title("Reliability diagram (test set)")
save("final_calibration.png")

Answer Here.

#### 7. Robustness: lighting, noise, blur and a covered mouth

In [ ]:
# Robustness: the same test faces with darker or brighter lighting, low contrast, noise, blur and a covered mouth
rs = np.random.RandomState(SEED)


def perturb(xu8, kind):
    x = xu8.astype(np.float32)
    mean = x.mean((1, 2), keepdims=True)
    if kind.startswith("dark"):
        x = x * 0.45
    elif kind.startswith("very dark"):
        x = x * 0.2
    elif kind == "bright (+90)":
        x = x + 90
    elif kind == "low contrast (x0.4)":
        x = (x - mean) * 0.4 + mean
    elif kind == "noise (sd 15)":
        x = x + rs.normal(0, 15, x.shape)
    elif kind == "blur (5x5 gaussian)":
        x = np.stack([cv2.GaussianBlur(a, (5, 5), 1.2) for a in x])
    elif kind == "mouth covered":
        x[:, 31:46, 8:40] = 128
    x = np.clip(x, 0, 255).astype(np.uint8)
    if kind.endswith("+ CLAHE"):
        x = np.stack([deepfer.equalize(a) for a in x])
    return x


rob_idx = te_idx[:600] if QUICK else te_idx
conds = ["original", "original + CLAHE", "dark (x0.45)", "dark (x0.45) + CLAHE", "very dark (x0.2)", "very dark (x0.2) + CLAHE",
         "bright (+90)", "low contrast (x0.4)", "noise (sd 15)", "blur (5x5 gaussian)", "mouth covered"]
rob = []
for kind in conds:
    sc = deepfer.scores(Y[rob_idx], final_logits(torch.from_numpy(perturb(X[rob_idx], kind))).argmax(1))
    rob.append({"condition": kind, "accuracy": sc["accuracy"], "macro F1": sc["macro F1"]})
rob = pd.DataFrame(rob).set_index("condition")
rob["accuracy drop"] = rob["accuracy"] - rob.loc["original", "accuracy"]
display(rob.round(4))
rob.round(4).to_csv("outputs/robustness.csv")

fig = plt.figure(figsize=(15, 4.4))
ax = fig.add_subplot(1, 2, 1)
rob["accuracy"].plot.barh(ax=ax, color=["#55A868" if "CLAHE" in c else "#4C72B0" for c in rob.index])
ax.axvline(rob.loc["original", "accuracy"], ls=":", c="grey"); ax.invert_yaxis(); ax.set_xlabel("test accuracy")
ax.set_title("Accuracy under changed conditions (green = with CLAHE)")
i0 = rob_idx[3]
for k, kind in enumerate(["original", "dark (x0.45)", "very dark (x0.2)", "very dark (x0.2) + CLAHE", "low contrast (x0.4)", "noise (sd 15)", "blur (5x5 gaussian)", "mouth covered"]):
    a = fig.add_subplot(2, 8, 5 + k + (k // 4) * 4)
    a.imshow(perturb(X[[i0]], kind)[0], cmap="gray", vmin=0, vmax=255); a.set_title(kind, fontsize=7); a.axis("off")
save("final_robustness.png")

Answer Here.

#### 8. Speed: is it fast enough for real time?

In [ ]:
# Speed: time to score one face, on this machine's GPU and on the CPU (batch of 1, the real-time case)
def time_ms(fn, n=40):
    for _ in range(5):
        fn()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1000


one = torch.from_numpy(X[te_idx[:1]])
lat = []
for name, m in NETS.items():
    row = {"model": name, "parameters (M)": deepfer.count_params(m) / 1e6,
           "file size (MB)": os.path.getsize(f"models/{FILES[name]}") / 1e6}
    for dev in ([DEVICE, torch.device("cpu")] if DEVICE.type != "cpu" else [DEVICE]):
        mm, _ = deepfer.load_model(f"models/{FILES[name]}", dev)     # a fresh copy from the saved file
        x1 = deepfer.to_batch(one, dev)

        def run():
            with torch.no_grad():
                mm(x1)
            deepfer.sync(dev)
        row[f"ms per face ({dev.type})"] = time_ms(run)
    lat.append(row)
lat = pd.DataFrame(lat).set_index("model")
fin = lat.loc[FINAL_MEMBERS].sum() * (2 if FINAL_TTA else 1)        # members run one after another, twice with TTA
lat.loc["final model"] = fin
lat.loc["final model", ["parameters (M)", "file size (MB)"]] = lat.loc[FINAL_MEMBERS, ["parameters (M)", "file size (MB)"]].sum()
display(lat.round(2))
lat.round(3).to_csv("outputs/latency.csv")

# face detection on a 640x480 frame (OpenCV Haar cascade on the CPU)
finder = deepfer.FaceFinder()
frame = cv2.resize(X[te_idx[0]], (640, 480))
det_ms = time_ms(lambda: finder.find(frame), n=20)
for col in [c for c in lat.columns if c.startswith("ms per face")]:
    total = det_ms + lat.loc["final model", col]
    print(f"{col}: detection {det_ms:.1f} ms + emotion {lat.loc['final model', col]:.1f} ms = {total:.1f} ms per frame "
          f"with one face, about {1000 / total:.0f} frames per second")

Answer Here.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Answer Here.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Answer Here.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Answer Here.

In [ ]:
# Explainability 1: Grad-CAM. Which parts of the face pushed the network to its answer?
# (shown for the first network of the final model on correctly classified test faces)
explain_net = net(FINAL_MEMBERS[0])
rs = np.random.RandomState(4)
fig, axes = plt.subplots(4, 7, figsize=(14, 8.4))
for k, c in enumerate(CLASSES):
    ok = np.where((y_te == LABEL_ID[c]) & (pred_te == y_te))[0]
    if len(ok) < 2:                                   # fall back to any faces of the emotion
        ok = np.where(y_te == LABEL_ID[c])[0]
    pick4 = rs.choice(ok, 2, replace=False)
    cams, _ = deepfer.grad_cam(explain_net, deepfer.to_batch(torch.from_numpy(X[te_idx[pick4]]), DEVICE))
    for r, (j, cam_) in enumerate(zip(pick4, cams)):
        axes[2 * r, k].imshow(X[te_idx[j]], cmap="gray"); axes[2 * r, k].axis("off")
        axes[2 * r + 1, k].imshow(X[te_idx[j]], cmap="gray"); axes[2 * r + 1, k].imshow(cam_, cmap="jet", alpha=0.45)
        axes[2 * r + 1, k].axis("off")
    axes[0, k].set_title(c)
plt.suptitle(f"Grad-CAM of {FINAL_MEMBERS[0]}: red = the regions that mattered most", y=0.95)
save("explain_gradcam.png")

# average Grad-CAM per emotion over 150 correct test faces each
avg = {}
for c in CLASSES:
    ok = np.where((y_te == LABEL_ID[c]) & (pred_te == y_te))[0][:150]
    if len(ok) == 0:
        ok = np.where(y_te == LABEL_ID[c])[0][:150]
    cams, _ = deepfer.grad_cam(explain_net, deepfer.to_batch(torch.from_numpy(X[te_idx[ok]]), DEVICE),
                               target=np.full(len(ok), LABEL_ID[c]))
    avg[c] = cams.mean(0)
fig, axes = plt.subplots(1, 7, figsize=(16, 2.8))
for ax, c in zip(axes, CLASSES):
    ax.imshow(overall, cmap="gray"); ax.imshow(avg[c], cmap="jet", alpha=0.5); ax.set_title(c); ax.axis("off")
plt.suptitle("Average Grad-CAM per emotion", y=1.05)
save("explain_gradcam_average.png")
for c in CLASSES:
    a = avg[c]
    print(f"{c:9s} share of attention: eyes band {a[10:24].sum() / (a.sum() + 1e-8):.0%}, mouth band {a[31:46].sum() / (a.sum() + 1e-8):.0%}")

Answer Here.

In [ ]:
# Explainability 2: occlusion. Cover one region of every test face with grey and see which emotions suffer.
occ_idx = te_idx[:600] if QUICK else te_idx
regions = {"forehead": (slice(0, 10), slice(0, 48)), "eyes and brows": (slice(10, 24), slice(0, 48)),
           "nose": (slice(22, 32), slice(12, 36)), "mouth": (slice(31, 46), slice(8, 40))}
base_pred = final_logits(torch.from_numpy(X[occ_idx])).argmax(1)
yo = Y[occ_idx]
base_rec = pd.Series([(base_pred[yo == i] == i).mean() for i in range(7)], index=CLASSES)
drop = {}
for rname, (rr, cc) in regions.items():
    xo = X[occ_idx].copy()
    xo[:, rr, cc] = 128
    p = final_logits(torch.from_numpy(xo)).argmax(1)
    drop[rname] = pd.Series([(p[yo == i] == i).mean() for i in range(7)], index=CLASSES) - base_rec
drop = pd.DataFrame(drop).T
fig, ax = plt.subplots(figsize=(10, 3.8))
sns.heatmap(drop * 100, annot=True, fmt=".0f", cmap="Reds_r", ax=ax, cbar_kws={"label": "change in recall (points)"})
ax.set_title("Recall change per emotion when one face region is hidden")
save("explain_occlusion.png")

Answer Here.

## ***7.1 Real-time processing: from a camera frame to an emotion***

Answer Here.

In [ ]:
# 1) Does the face detector find the test faces, and do its crops give the same answers as the original crops?
# Every test face is enlarged to 160x160 and put on a grey background, like a face in a photo.
finder = deepfer.FaceFinder()
rs = np.random.RandomState(SEED)
rt_idx = rs.choice(te_idx, 300 if QUICK else 2000, replace=False)
found, crops, keep_i = 0, [], []
for i in rt_idx:
    big = cv2.copyMakeBorder(cv2.resize(X[i], (160, 160), interpolation=cv2.INTER_CUBIC), 48, 48, 48, 48, cv2.BORDER_CONSTANT, value=128)
    boxes = finder.find(big)
    if len(boxes) == 1:
        crops.append(deepfer.crop_face(big, boxes[0])); keep_i.append(i)
keep_i = np.array(keep_i)
acc_orig = (final_logits(torch.from_numpy(X[keep_i])).argmax(1) == Y[keep_i]).mean()
acc_det = (final_logits(torch.from_numpy(np.stack(crops))).argmax(1) == Y[keep_i]).mean()
print(f"faces found by the detector: {len(keep_i)} of {len(rt_idx)} ({len(keep_i) / len(rt_idx):.0%})")
print(f"accuracy on those faces: original crops {acc_orig:.3f}, crops made by the real-time pipeline {acc_det:.3f}")

# 2) A whole frame with several faces through the same pipeline the app and the webcam script use
class NotebookPredictor:                              # the final model in memory, same interface as deepfer.EmotionPredictor
    def predict(self, faces):
        if len(faces) == 0:
            return np.zeros((0, 7))
        return softmax(final_logits(torch.from_numpy(np.stack(faces))) / T)


rs = np.random.RandomState(11)
canvas = np.full((480, 800), 128, np.uint8)
spots = [(40, 50, 170), (300, 40, 150), (560, 60, 190), (60, 270, 150), (320, 250, 180), (590, 290, 140)]
truth = []
for (x0, y0, s), c in zip(spots, rs.permutation(CLASSES)[:6]):
    i = rs.choice(te_idx[y_te == LABEL_ID[c]])            # random test faces, right or wrong
    canvas[y0:y0 + s, x0:x0 + s] = cv2.resize(X[i], (s, s), interpolation=cv2.INTER_CUBIC)
    truth.append(c)
predictor_nb = NotebookPredictor()
t_det = time_ms(lambda: finder.find(canvas), n=10)
boxes, faces, probs = deepfer.analyse_image(canvas, finder, predictor_nb)
t_cls = time_ms(lambda: predictor_nb.predict(faces), n=10)
print(f"faces found {len(boxes)} of 6 | detection {t_det:.1f} ms + emotions for all faces {t_cls:.1f} ms "
      f"= {t_det + t_cls:.1f} ms per frame (about {1000 / (t_det + t_cls):.0f} frames per second)")
show_img = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
for (x, y, w, h), p in zip(boxes, probs):
    cv2.rectangle(show_img, (x, y), (x + w, y + h), (0, 200, 0), 2)
    cv2.putText(show_img, f"{CLASSES[p.argmax()]} {p.max():.0%}", (x, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
plt.figure(figsize=(10, 6)); plt.imshow(show_img); plt.axis("off")
plt.title("Real-time pipeline on a test frame (true emotions: " + ", ".join(truth) + ")")
save("realtime_demo_frame.png")
cv2.imwrite("outputs/demo_frame.png", canvas)

Answer Here.

## ***7.2 Application: Streamlit app and live webcam window***

Answer Here.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
# PyTorch models are saved with torch.save (the PyTorch equivalent of a pickle file): the weights plus the
# settings needed to rebuild the network. models/model_card.json says which networks make up the final model,
# whether the mirror trick is used, the temperature and the test scores, so the app loads exactly this model.
card = {"members": [FILES[n] for n in FINAL_MEMBERS], "member_names": FINAL_MEMBERS, "tta": FINAL_TTA,
        "temperature": T, "classes": CLASSES, "input": "48x48 grayscale face, values 0-255 (the model scales them)",
        "test_scores": {k.replace("test ", ""): round(v, 4) for k, v in RESULTS[FINAL_NAME].items() if k.startswith("test")},
        "val_scores": {k.replace("val ", ""): round(v, 4) for k, v in RESULTS[FINAL_NAME].items() if k.startswith("val")},
        "ms_per_face": {c: round(float(v), 2) for c, v in lat.loc["final model"].items() if c.startswith("ms")},
        "trained_on": str(DEVICE), "pixel_mean": PIXEL_MEAN, "pixel_std": PIXEL_STD}
Path("models/model_card.json").write_text(json.dumps(card, indent=2))

comparison = pd.DataFrame(RESULTS).T
comparison.round(4).to_csv("outputs/model_comparison.csv")
for name, h in [("cnn_first", hist_cnn0), ("cnn_tuned", hist_cnn)] + [(k, v) for k, v in TL_HIST.items()]:
    h.to_csv(f"outputs/history_{name}.csv", index=False)

# a few test faces for trying the app (file name = true emotion)
sample_dir = Path("outputs/sample_faces")
sample_dir.mkdir(parents=True, exist_ok=True)
rs = np.random.RandomState(5)
for c in CLASSES:
    for k, i in enumerate(rs.choice(te_idx[y_te == LABEL_ID[c]], 4, replace=False)):
        cv2.imwrite(str(sample_dir / f"{c}_{k + 1}.png"), X[i])
for f in sorted(Path("models").iterdir()):
    print(f"{f}  {f.stat().st_size / 1e6:.2f} MB")
display(comparison[[c for c in comparison.columns if c.startswith("test")]].round(4))

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
# A fresh copy of the deployed model, loaded from the files only (exactly what the app does), on the CPU.
importlib.reload(deepfer)
loaded = deepfer.EmotionPredictor("models", device=torch.device("cpu"))
print("loaded:", loaded.card["member_names"], "| mirror TTA:", loaded.tta, "| temperature:", round(loaded.temperature, 2))

# the sample test faces saved above: the model never saw them during training or model choice
files = sorted(Path("outputs/sample_faces").glob("*.png"))
faces = [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in files]
p = loaded.predict(faces)
res = pd.DataFrame({"file": [f.name for f in files], "true": [f.stem.rsplit("_", 1)[0] for f in files],
                    "predicted": [CLASSES[k] for k in p.argmax(1)], "confidence": p.max(1).round(3)})
display(res)
print(f"{(res.true == res.predicted).mean():.0%} of these {len(res)} sample faces are right")

# the same answers as the model in memory?
gap = np.abs(p - softmax(final_logits(torch.from_numpy(np.stack(faces))) / T)).max()
print(f"largest difference to the model in memory: {gap:.4f} (the GPU run uses half precision, the CPU full precision)")
print("networks reloaded because their GPU copy had changed:", dict(pd.Series(RELOADS).value_counts()) if RELOADS else "none")

# and a whole frame from disk through the real-time pipeline
frame = cv2.imread("outputs/demo_frame.png", cv2.IMREAD_GRAYSCALE)
boxes, faces_f, probs_f = deepfer.analyse_image(frame, deepfer.FaceFinder(), loaded)
print("faces in the demo frame:", [(b, CLASSES[q.argmax()], round(float(q.max()), 2)) for b, q in zip(boxes, probs_f)])

### 3. Download the results (only on Colab)

In [ ]:
# On Colab: put the trained models, the figures, the result tables and deepfer.py into one zip and download it,
# so the Streamlit app and the webcam script can use them on a laptop. (Nothing happens when run locally.)
if "google.colab" in sys.modules:
    from google.colab import files
    with zipfile.ZipFile("deepfer_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("deepfer.py")
        for folder in ["models", "outputs"]:
            for f in sorted(Path(folder).rglob("*")):
                if f.is_file():
                    z.write(f)
    print(f"deepfer_results.zip: {os.path.getsize('deepfer_results.zip') / 1e6:.1f} MB")
    files.download("deepfer_results.zip")
else:
    print("running locally: the models and outputs are already in the project folder")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

Answer Here.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***